# Cross-encoder reranker

Fine-tunes `ncbi/MedCPT-Cross-Encoder` on TermMap's out-of-fold candidate groups, then scores
request files. Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

All files are read from and written to `My Drive/HiLabs_Clinical`.

**Settings** (first code cell): `RUN` selects the training file and the request files.

| RUN | Train on | Score | Outputs |
|---|---|---|---|
| `dev` | `ce_train_dev.jsonl` | `ce_requests_dev.jsonl` | `outputs/ce_scores_dev.jsonl` |
| `full` | `ce_train_full.jsonl` | `ce_requests_<name>.jsonl` | `outputs/ce_scores_<name>.jsonl` |

Each training group holds the term (with its context), the gold description first and up to 15
hard negatives taken from the ranker's own candidate list. The loss is a softmax cross-entropy over
the group. The fine-tuned model is saved to `cache/ce_<RUN>/` and reused if present.

In [ ]:
RUN = "dev"                      # "dev" or "full"
SCORE_NAMES = ["dev"]            # for RUN="full", the suffix of ce_requests_<name>.jsonl, e.g. ["eval"]
BASE = "ncbi/MedCPT-Cross-Encoder"
EPOCHS = 2
GROUPS_PER_BATCH = 4             # x 16 documents
MAX_LEN = 192
LR = 2e-5
SEED = 13

In [ ]:
!pip -q install -U transformers accelerate

In [ ]:
import json, math, os, random, time, numpy as np, torch
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/HiLabs_Clinical"
assert os.path.isdir(WORK), f"create the folder {WORK} in Google Drive first"
CACHE, OUT = f"{WORK}/cache", f"{WORK}/outputs"
os.makedirs(CACHE, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "switch the runtime to a GPU"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_DIR = f"{CACHE}/ce_{RUN}"

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

train_path = f"{WORK}/ce_train_{RUN}.jsonl"
req_paths = {n: f"{WORK}/ce_requests_{n}.jsonl" for n in SCORE_NAMES}
for p in [train_path, *req_paths.values()]:
    assert os.path.exists(p) or os.path.exists(f"{MODEL_DIR}/config.json"), f"missing {p}"
print("model dir:", MODEL_DIR)

In [ ]:
if os.path.exists(f"{MODEL_DIR}/config.json"):
    print("loading fine-tuned model from", MODEL_DIR)
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).cuda()
else:
    groups = read_jsonl(train_path)
    print(len(groups), "training groups")
    tok = AutoTokenizer.from_pretrained(BASE)
    model = AutoModelForSequenceClassification.from_pretrained(BASE).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(groups) / GROUPS_PER_BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.cuda.amp.GradScaler()
    model.train()
    step, t = 0, time.time()
    for epoch in range(EPOCHS):
        order = list(range(len(groups)))
        random.Random(SEED + epoch).shuffle(order)
        for b in range(0, len(order), GROUPS_PER_BATCH):
            batch = [groups[i] for i in order[b:b + GROUPS_PER_BATCH]]
            q = [g["query"] for g in batch for _ in g["docs"]]
            d = [doc for g in batch for doc in g["docs"]]
            sizes = [len(g["docs"]) for g in batch]
            enc = tok(q, d, truncation="longest_first", max_length=MAX_LEN, padding=True, return_tensors="pt").to("cuda")
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc).logits.squeeze(-1)
            loss, pos = 0.0, 0
            for n in sizes:   # gold is the first document of each group
                loss = loss + torch.nn.functional.cross_entropy(logits[pos:pos + n].float().unsqueeze(0),
                                                                torch.zeros(1, dtype=torch.long, device="cuda"))
                pos += n
            loss = loss / len(sizes)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            step += 1
            if step % 100 == 0:
                print(f"epoch {epoch + 1} step {step}/{steps} loss {loss.item():.4f} {time.time() - t:.0f}s", flush=True)
    model.save_pretrained(MODEL_DIR)
    tok.save_pretrained(MODEL_DIR)
    print("saved", MODEL_DIR)

In [ ]:
model.eval()

@torch.no_grad()
def score(reqs, batch=128):
    pairs = [(i, r["query"], d) for i, r in enumerate(reqs) for d in r["docs"]]
    logits = []
    for b in range(0, len(pairs), batch):
        chunk = pairs[b:b + batch]
        enc = tok([p[1] for p in chunk], [p[2] for p in chunk], truncation="longest_first",
                  max_length=MAX_LEN, padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            logits += model(**enc).logits.squeeze(-1).float().cpu().tolist()
    out, pos = [], 0
    for r in reqs:
        n = len(r["docs"])
        p = torch.softmax(torch.tensor(logits[pos:pos + n]), dim=0).tolist()
        out.append({"id": r["id"], "probs": [round(x, 6) for x in p]})
        pos += n
    return out

for name, p in req_paths.items():
    reqs = read_jsonl(p)
    scores = score(reqs)
    path = f"{OUT}/ce_scores_{name}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for s in scores:
            fh.write(json.dumps(s) + "\n")
    print(name, len(scores), "->", path)